<a href="https://colab.research.google.com/github/bearsmithnu/bearsmithnu/blob/main/hw4/embeddings-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Language Word Embeddings

In class, we discussed how we can reduce the dimensionality of word representations from their original vector space to an embedding space on the order of a few hundred dimensions. Different modeling choices for word embeddings may be ultimately evaluated by the effectiveness of classification or retrieval models.

In this assignment, however, we will consider another common method of evaluating word embeddings: by judging the usefulness of pairwise distances between words in the embedding space.

Follow along with the examples in this notebook, and implement the sections of code flagged with **TODO**.

In [1]:
## If you already have gensim, you can comment this out.
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.2 MB/s  0:00:00


In [2]:
import gensim
import numpy as np
from gensim.test.utils import datapath, get_tmpfile
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

We'll start by downloading a plain-text version of the plays of William Shakespeare.

In [3]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/shakespeare_plays.txt
lines = [s.split() for s in open('shakespeare_plays.txt')]

--2026-11-22 15:49:09--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/shakespeare_plays.txt
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4746840 (4.5M) [text/plain]
Saving to: ‘shakespeare_plays.txt’

shakespeare_plays.t 100%[===================>]   4.53M  30.0MB/s    in 0.2s    

2026-11-22 15:49:09 (30.0 MB/s) - ‘shakespeare_plays.txt’ saved [4746840/4746840]



Then, we'll estimate a simple word2vec model on the Shakespeare texts.

In [6]:
model = Word2Vec(lines)

Even with such a small training set size, you can perform some standard analogy tasks we discussed in class.

In [7]:
model.wv.most_similar(positive=['king','woman'], negative=['man'])

[('queen', 0.8102731704711914),
 ('gloucester', 0.7199225425720215),
 ('warwick', 0.7197041511535645),
 ('prince', 0.7193968296051025),
 ('york', 0.7149473428726196),
 ('clarence', 0.7007802128791809),
 ('duke', 0.6906498074531555),
 ('northumberland', 0.6736332774162292),
 ('son', 0.6670017838478088),
 ('cardinal', 0.6614840030670166)]

In other words, we want a vector close to `king` and `woman` but subtracting the dimensions that are important to `man`, i.e., `queen`. Other words are mostly noble titles important in Shakespeare's "history" plays.

For the rest of this assignment, we will focus on finding words with similar embeddings, both within and across languages. For example, what words are similar to the name of the title character of *Othello*?

In [10]:
model.wv.most_similar(positive=['othello'])
#model.wv.most_similar(positive=['brutus'])

[('iago', 0.9166520833969116),
 ('desdemona', 0.9160134792327881),
 ('cleopatra', 0.905907928943634),
 ('pisanio', 0.8985666632652283),
 ('imogen', 0.8960666656494141),
 ('jul', 0.8830298781394958),
 ('emilia', 0.8749774098396301),
 ('rom', 0.874350368976593),
 ('lucio', 0.8699493408203125),
 ('charmian', 0.8693265914916992)]

If you know the play, you might see some familiar names.

This search uses cosine similarity. In the default API, you should see the same similarity between the words `othello` and `desdemona` as in the search results above.

In [14]:
model.wv.similarity('othello', 'desdemona')

np.float32(0.91601354)

**TODO**: Your **first task**, therefore, is to implement your own cosine similarity function so that you can reuse it outside of the context of the gensim model object.

In [63]:
## TODO: Implement cosim
def cosim(v1, v2):
  ## return cosine similarity between v1 and v2
  return np.dot(v1, v2) / (np.dot(v1, v1)**0.5 * np.dot(v2, v2)**0.5)

## This should give a result similar to model.wv.similarity:
cosim(model.wv['othello'], model.wv['desdemona'])

np.float32(0.9160134)

## Evaluation

We could collect a lot of human judgments about how similar pairs of words, or pairs of Shakespearean characters, are. Then we could compare different word-embedding models by their ability to replicate these human judgments.

If we extend our ambition to multiple languages, however, we can use a word translation task to evaluate word embeddings.

We will use a subset of [Facebook AI's FastText cross-language embeddings](https://fasttext.cc/docs/en/aligned-vectors.html) for several languages. Your task will be to compare English both to French, and to *one more language* from the following set: Arabic, German, Portuguese, Russian, Spanish, Vietnamese, and Chinese.

In [17]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.en.vec
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.fr.vec

# TODO: uncomment at least one of these to work with another language
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.ar.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.de.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.pt.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.ru.vec
# español :)
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.es.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.vi.vec
# !wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.zh.vec

--2026-11-22 16:10:38--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.en.vec
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67681172 (65M)
Saving to: ‘30k.en.vec’

30k.en.vec          100%[===================>]  64.54M  60.0MB/s    in 1.1s    

2026-11-22 16:10:40 (60.0 MB/s) - ‘30k.en.vec’ saved [67681172/67681172]

--2026-11-22 16:10:40--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/30k.fr.vec
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67802327 (65M)
Saving to: ‘30k.fr.vec’

30k.fr.vec          100%[===================>]  64.66M  45.6MB/s    in 1.4s    

2026-11-22 16:10:41 (45.6 MB/s) - ‘30k.fr.vec’ saved [67802327/67802327]

--2026-11-22 16:10:41--  http://www.ccs.neu.

We'll start by loading the word vectors from their textual file format to a dictionary mapping words to numpy arrays.

In [18]:
def vecref(s):
  (word, srec) = s.split(' ', 1)
  return (word, np.fromstring(srec, sep=' '))

def ftvectors(fname):
  return { k:v for (k, v) in [vecref(s) for s in open(fname)] if len(v) > 1}

envec = ftvectors('30k.en.vec')
frvec = ftvectors('30k.fr.vec')

# TODO: load vectors for one more language, such as zhvec (Chinese)
# arvec = ftvectors('30k.ar.vec')
# devec = ftvectors('30k.de.vec')
# ptvec = ftvectors('30k.pt.vec')
# ruvec = ftvectors('30k.ru.vec')
esvec = ftvectors('30k.es.vec')
# vivec = ftvectors('30k.vi.vec')
# zhvec = ftvectors('30k.zh.vec')

**TODO**: Your next task is to write a simple function that takes a vector and a dictionary of vectors and finds the most similar item in the dictionary. For this part of the assignment, a linear scan through the dictionary using your `cosim` function from above is acceptible.

In [64]:
## TODO: implement this search function
def mostSimilar(vec, vecDict):
  ## Use your cosim function from above
  mostSimilar = ''
  similarity = 0
  for term, termVec in vecDict.items():
    thisSim = cosim(vec, termVec)
    if thisSim > similarity:
      similarity = thisSim
      mostSimilar = term
  return (mostSimilar, similarity)

## some example searches
[mostSimilar(envec[e], frvec) for e in ['computer', 'germany', 'matrix', 'physics', 'yeast']]

[('informatique', np.float64(0.5023827767603765)),
 ('allemagne', np.float64(0.593718413875964)),
 ('matrice', np.float64(0.5088361302065517)),
 ('physique', np.float64(0.4555543434796394)),
 ('fermentation', np.float64(0.3504105196166514))]

Some matches make more sense than others. Note that `computer` most closely matches `informatique`, the French term for *computer science*. If you looked further down the list, you would see `ordinateur`, the term for *computer*. This is one weakness of a focus only on embeddings for word *types* independent of context.

To evalute cross-language embeddings more broadly, we'll look at a dataset of links between Wikipedia articles.

In [31]:
!wget http://www.ccs.neu.edu/home/dasmith/courses/cs6200/links.tab
links = [s.split() for s in open('links.tab')]

--2026-11-22 16:20:48--  http://www.ccs.neu.edu/home/dasmith/courses/cs6200/links.tab
Resolving www.ccs.neu.edu (www.ccs.neu.edu)... 52.70.229.197
Connecting to www.ccs.neu.edu (www.ccs.neu.edu)|52.70.229.197|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1408915 (1.3M)
Saving to: ‘links.tab’

links.tab           100%[===================>]   1.34M  --.-KB/s    in 0.1s    

2026-11-22 16:20:48 (11.9 MB/s) - ‘links.tab’ saved [1408915/1408915]



This `links` variable consists of triples of `(English term, language, term in that language)`. For example, here is the link between English `academy` and French `académie`:

In [32]:
links[302]

['academy', 'fr', 'académie']

We construct a test set for English-French from the first 1000 links between those languages.

In [33]:
frtest = [x for x in links if x[1] == "fr"][0:1000]
frtest[0:10]

[['aalborg', 'fr', 'aalborg'],
 ['aarhus', 'fr', 'aarhus'],
 ['aba', 'fr', 'aba'],
 ['abad', 'fr', 'abad'],
 ['abandon', 'fr', 'abandon'],
 ['abbas', 'fr', 'abbas'],
 ['abbasid', 'fr', 'abbassides'],
 ['abbess', 'fr', 'abbesse'],
 ['abbey', 'fr', 'abbaye'],
 ['abbot', 'fr', 'abbé']]

**TODO**: Evaluate the English and French embeddings by computing the proportion of English Wikipedia articles whose corresponding French article in this test set `frtest` is also the closest word in embedding space. Skip English articles not covered by the word embedding dictionary.

Since many articles, e.g., about named entities, have the same title in English and French, use the identity function as a baseline and compute its accuracy. In other words, how often would you find the right French articles by simply echoing the English title as if it were French? In the ten example links above, `aalborg` and `aarhus` (two cities in Denmark) are the same in English and French. Remember to iterate only over the 1000 linked Wikipedia articles in the test set, not the entire embedding dictionary.

In [46]:
## TODO: Compute English-French Wikipedia retrieval accuracy.

def retrieval_acc(test_set, foreign_wv):
  # Initialize counters for the observed number of matches between English and
  # French embeddings, the baseline matches where the English and French terms
  # are identical, and the number of comparisons made.
  real_match = 0
  baseline_match = 0
  compared_count = 0

  # Count the observed and baseline matches between the two languages.
  for row in test_set:
    en_term = row[0]
    foreign_term = row[2]
    en_vec = envec.get(en_term, None)
    # If there is no English embedding, skip this iteration.
    if en_vec is None:
      continue

    # If the English and foreign terms match, increment the baseline accuracy.
    if en_term == foreign_term:
      baseline_match += 1

    # Find the most similar foreign term to the English vector.
    most_sim, score = mostSimilar(en_vec, foreign_wv)
    if most_sim == foreign_term:
      real_match += 1

    # Increment the number of comparisons made.
    compared_count += 1

  # Calculate the observed and baseline accuracies and return them.
  return real_match / compared_count, baseline_match / compared_count

accuracy, baselineAccuracy = retrieval_acc(frtest, frvec)

In [49]:
print(accuracy)
print(baselineAccuracy)

0.575
0.661


**TODO**: Compute the accuracy, i.e. precision@1, of the embeddings and of the identity function for the first 1000 links between English and another language besides French. Although the baseline will be lower for languages not written in the Roman alphabet (i.e., Arabic or Chinese), there are still many articles in those languages with headwords written in Roman characters.

In [65]:
## TODO: Compute English-X Wikipedia retrieval accuracy.
estest = [x for x in links if x[1] == "es"][0:1000]

retrieval_acc(estest, esvec)
# Note: These results are cool in the difference they reveal between Spanish
# and French and their relationships to English! Spanish has a lower baseline
# accuracy, which I imagine is the result of them being more prone to convert
# loanwords into Spanish phonetic spelling- e.g. tea vs. té, hamburger vs.
# hamburguesa, while it seems like French just takes them wholesale. (I don't
# know French, admittedly.) Spanish has a higher retrieval accuracy, though,
# and I wonder if that speaks to the similarity in structure between it and
# English. When I was first learning Spanish, I was shocked how frequently I
# could translate a sentence nearly word-for-word.

(0.536, 0.518)

**TODO**: Find the 10 nearest neighbors of each English term to compute "recall at 10" and "mean reciprocal rank at 10".

In [89]:
## TODO: Compute recall@10 and MRR@10 when retrieving 10 nearest neighbors in French and some other language.

# Modification of the mostSimilar function from above to retrieve the 10
# terms from the dictionary with highest cosine similarity to the one given.
def get_ten_nearest(vec, vecDict):
  # Store the running ten nearest neighbors.
  most_similar = [("", -1.0)] * 10
  for term, termVec in vecDict.items():
    thisSim = cosim(vec, termVec)
    # If this similarity is greater than one of the top ten, insert it
    # and remove the final item.
    for i in range(9, -1, -1):
      if thisSim > most_similar[i][1]:
        most_similar.insert(i, (term, thisSim))
        most_similar.pop(10)

  return (most_similar)


def recall_mrr_at_10(test_set, foreign_vec):
  # Initialize counters for the number of successful recalls, an accumulator
  # for reciprocal rank, and the number of comparisons made.
  num_recalled = 0
  rr_sum = 0
  compared_count = 0

  for row in test_set:
    en_term = row[0]
    foreign_term = row[2]
    en_vec = envec.get(en_term, None)

    # If there is no English embedding, skip this iteration.
    if en_vec is None:
      continue

    # Get the ten nearest-neighbors in the foreign language. If they
    # contain our foreign term, increment the recall counter and add the
    # reciprocal rank, then continue to the next row.
    ten_nearest = get_ten_nearest(en_vec, foreign_vec)
    for i in range (10):
      if ten_nearest[i][0] == foreign_term:
        num_recalled += 1
        rr_sum += 1/(i + 1)
        break

    # Note that we searched for this term.
    compared_count += 1

  # Calculate the average recall @ 10 and the mean reciprocal rank.
  return num_recalled / compared_count, rr_sum / compared_count


print(recall_mrr_at_10(frtest, frvec))
print(recall_mrr_at_10(estest, esvec))

(0.636, 0.5989027777777777)
done with french
(0.604, 0.5619468253968254)


## Speeding up Vector Search (required for CS6200, 20 points extra for CS4220)

The list of Wikipedia headwords is short enough that a linear scan through the non-English language embeddings takes some time but is feasible. In a production system, you could index the word embeddings using SimHash or some other locality sensitive hashing scheme, as we discussed for duplicate detection, to speed up this process.

A relatively easy way to get started with fast vector similarity search is to install Meta's `faiss` (Facebook AI Similarity Search) package and read [the tutorial](https://github.com/facebookresearch/faiss/wiki/Getting-started).

In [ ]:
# Outside of colab, you may need a different package manager.
# !apt install libomp-dev
!pip install --upgrade faiss-cpu
# Use this line instead if you have a GPU.
# !python -m pip install --upgrade faiss-gpu
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 56.7 MB/s  0:00:00


**TODO**: Create two vector indexes, for the FastText embeddings of French and for the other language you evaluated above. Use `faiss` to find the 10 nearest neighbors for the top 1000 Wikipedia headwords you evaluated for English-French and the English-X as above.

In [ ]:
## TODO: Create two FAISS indexes for the embeddings in French
##       and the other non-English language of your choice.


First, measure the _effectiveness_ of this approximate vector search approach. How does the R@10 and MRR@10 using `faiss` compare to the brute-force search you did above?

In [ ]:
## TODO: Measure the R@10 and MRR@10 of the test set for the two languages.
##       Compute the difference in effecticeness to your brute-force search above.

Second, measure the _efficiency_ of this approach. How long in seconds does finding nearest neighbors for 1000 headwords by brute force compare to using `faiss`? (For this exercise, don't worry about amortizing indexing costs.)

In [ ]:
## TODO: Find the 1000 nearest heawords in French and the other language for each query in the test set.
##       Compare the retrieval time in seconds to the speed of your brute-force search.
##       You shouldn't include the time to create the FAISS index.